# QC: типы карт на дашборде — через витрину, не через `scd1_trx`

`ods_alpha.scd1_trx` — миллиарды строк. **Этот скан в тетрадке выключен.**  
На дашборд тип карты из trx в рантайме тоже не тащим.

Проверяем, что уже лежит в **прогруженной витрине** (то же зерно, что дашборд):

`sbx_da.tmp_shestopalov_acq_datamart_final_script_2`

| Вопрос | Зачем |
|---|---|
| Есть ли колонка типа карты / FIID / платёжной системы? | Можно ли сразу нарисовать разбивку |
| Какое зерно у `final_df`? | Договор × месяц, не транзакция |
| Что можно спутать с типом карты? | `business_cards`, `mcc` — это не бренд карты |
| Есть ли соседняя trx-витрина как у MCC? | Паттерн для отдельной таблицы `card_type × month` |

Если LDAP DRP снова не пустит — тетрадка возьмёт локальный CSV `final_df` из `DATA_DIR`.

Справочник FIID (`scd1_base24_fiids`) — маленькая таблица, опционально.  
`scd1_trx` **не** читаем.

Новый kernel нормален.


In [ ]:
import re
from getpass import getpass
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 220)
pd.set_option('display.max_rows', 80)

DATA_DIR = Path('/home/jovyan/documents/Equaring/Data')
OUT_DIR = DATA_DIR / 'qc_card_types_fiid'
OUT_DIR.mkdir(parents=True, exist_ok=True)

DRP_DATAMART = 'sbx_da.tmp_shestopalov_acq_datamart_final_script_2'
DRP_MCC_TABLE = 'sbx_da.tmp_shestopalov_acq_mcc_month'
DRP_USER_DEFAULT = 'Shestopalov-VYur'

# Если DRP не зайдёт — CSV витрины (тот же final_df, что заливали).
FINAL_DF_CSV_CANDIDATES = [
    DATA_DIR / 'final_df_period_2026_01_2026_08_final_script_2.csv',
    DATA_DIR / 'final_df_period_2026_01_2026_07_final_script_2.csv',
    DATA_DIR / 'final_df_period_2026_01_2026_07_mpos.csv',
]

# Справочник FIID маленький. True = «типы в принципе есть в озере».
# scd1_trx не трогаем.
RUN_FIID_CATALOG = True
FIID_TABLE = 'ods_alpha.scd1_base24_fiids'
MEM_LIMIT = '4g'

CARD_COL_NEEDLES = (
    'card_type', 'cardtype', 'card_brand', 'payment_system', 'paysystem',
    'c_fiid_desc', 'c_fiid_iss', 'fiid_iss', 'fiid_desc',
    'visa', 'mastercard', 'мир', 'mir', 'unionpay', 'brand_ps',
)
CONFUSABLE = {
    'business_cards': 'продукт клиента «Бизнес-карта», не Visa/MC/МИР на транзакции',
    'pravocard': 'продукт Правокард, не бренд карты',
    'mcc': 'MCC точки / мерчанта, не тип карты',
    'mcc_profile': 'профиль MCC договора, не тип карты',
}

NOTEBOOK_REV = '2026-09-18-card-types-datamart-v2'
print('rev', NOTEBOOK_REV)
print('datamart', DRP_DATAMART)
print('trx scan: OFF')


## 0) Helpers


In [ ]:
def _is_ldap(exc):
    s = str(exc).lower()
    return 'ldap' in s or 'authentication failed' in s or 'password' in s and 'auth' in s


def fetch_drp(sql, label, allow_fail=False):
    t0 = pd.Timestamp.now()
    print(f'FETCH DRP {label} ...')
    try:
        with drp:
            df = drp.fetch(sql)
        if df is None:
            df = pd.DataFrame()
        print(f'  rows={len(df):,}  {(pd.Timestamp.now() - t0).total_seconds():.1f}s')
        return df
    except Exception as exc:
        print(f'  FAIL {type(exc).__name__}: {exc}')
        if allow_fail:
            return None
        raise


def fetch_imp(sql, label, allow_fail=False):
    t0 = pd.Timestamp.now()
    print(f'FETCH IMP {label} ...')
    try:
        with imp:
            imp.execute(f'set MEM_LIMIT={MEM_LIMIT}')
            df = imp.fetch(sql)
        if df is None:
            df = pd.DataFrame()
        print(f'  rows={len(df):,}  {(pd.Timestamp.now() - t0).total_seconds():.1f}s')
        return df
    except Exception as exc:
        print(f'  FAIL {type(exc).__name__}: {exc}')
        if allow_fail:
            return None
        raise


def normalize_month(s):
    ts = pd.to_datetime(s, errors='coerce')
    if pd.isna(ts):
        raw = str(s).strip()
        return raw[:7] if len(raw) >= 7 else None
    return ts.strftime('%Y-%m')


def classify_brand(text):
    s = '' if pd.isna(text) else str(text).strip().lower()
    if not s or s in {'none', 'nan', 'null', 'unknown', ''}:
        return 'empty'
    if re.search(r'union\s*pay|unionpay|юнион', s):
        return 'UnionPay'
    if re.search(r'\bmir\b|мир|nspk|нспк', s):
        return 'МИР'
    if re.search(r'master\s*card|mastercard|\bmaestro\b', s):
        return 'Mastercard'
    if re.search(r'\bvisa\b|виза', s):
        return 'Visa'
    if re.search(r'\bjcb\b', s):
        return 'JCB'
    if re.search(r'amex|american\s*express', s):
        return 'Amex'
    return 'other'


def score_card_col(name):
    n = re.sub(r'[^a-zа-я0-9]+', '', str(name).lower())
    hits = [k for k in CARD_COL_NEEDLES if re.sub(r'[^a-zа-я0-9]+', '', k) in n]
    return hits


print('helpers ready')


## 1) Витрина дашборда из DRP

Та же таблица, что заливает `final_script_2`. Если сессия DRP уже есть в kernel — переиспользуем.


In [ ]:
source_kind = None
fd = pd.DataFrame()
datamart_cols = pd.DataFrame()
drp_ok = False

if 'drp' in globals() and drp is not None:
    print('Reuse existing DRP')
    drp_ok = True
else:
    try:
        drp_user = input('DRP user: ').strip() or DRP_USER_DEFAULT
        drp_password = getpass('DRP password: ')
        drp = connect(
            to='DRP',
            user_params={'user_name': drp_user, 'password': drp_password},
        )
        ping = fetch_drp('SELECT current_user AS u, now() AS ts', 'ping')
        display(ping)
        drp_ok = True
        print('DRP AUTH OK as', drp_user)
    except Exception as exc:
        print('DRP connect failed:', type(exc).__name__, exc)
        if _is_ldap(exc):
            print('LDAP как раньше: витрину читаем из CSV, если файл есть.')
        drp = None
        drp_ok = False

if drp_ok:
    schema, table = DRP_DATAMART.split('.', 1)
    datamart_cols = fetch_drp(
        f'''
        SELECT column_name, data_type
        FROM information_schema.columns
        WHERE lower(table_schema) = lower('{schema}')
          AND lower(table_name) = lower('{table}')
        ORDER BY ordinal_position
        ''',
        'datamart columns',
        allow_fail=True,
    )
    fd = fetch_drp(f'SELECT * FROM {DRP_DATAMART}', 'datamart full', allow_fail=True)
    if fd is not None and len(fd):
        source_kind = 'drp'
        print('Источник: DRP', DRP_DATAMART, 'rows=', f'{len(fd):,}')
    else:
        print('DRP таблица пустая или SELECT не прошёл — пробуем CSV.')

if source_kind is None:
    csv_path = next((p for p in FINAL_DF_CSV_CANDIDATES if p.exists()), None)
    if csv_path is None:
        raise RuntimeError(
            'Нет витрины: DRP недоступна и нет CSV final_df в DATA_DIR. '
            + ', '.join(p.name for p in FINAL_DF_CSV_CANDIDATES)
        )
    fd = pd.read_csv(csv_path, dtype=str, low_memory=False)
    source_kind = 'csv'
    print('Источник: CSV', csv_path, 'rows=', f'{len(fd):,}')

fd.columns = [str(c).strip() for c in fd.columns]
print('cols', len(fd.columns))


## 2) Есть ли в витрине тип карты?

Ищем колонки вроде `payment_system`, `c_fiid_desc`, `card_type`.  
`business_cards` и `mcc` помечаем как **не** тип карты.


In [ ]:
if datamart_cols is None or datamart_cols.empty:
    datamart_cols = pd.DataFrame({
        'column_name': list(fd.columns),
        'data_type': ['from_data'] * len(fd.columns),
    })

col_audit = []
for _, r in datamart_cols.iterrows():
    name = str(r['column_name'])
    hits = score_card_col(name)
    conf = CONFUSABLE.get(name.lower())
    col_audit.append({
        'column_name': name,
        'data_type': r.get('data_type'),
        'card_type_candidate': int(bool(hits) and not conf),
        'needles': ', '.join(hits) if hits else '',
        'note': conf or ('похоже на тип карты' if hits else ''),
    })
col_audit = pd.DataFrame(col_audit)
card_cols = col_audit.loc[col_audit['card_type_candidate'] == 1, 'column_name'].tolist()
conf_cols = [c for c in fd.columns if str(c).lower() in CONFUSABLE]

print('=== колонки, похожие на тип карты ===')
if card_cols:
    display(col_audit[col_audit['card_type_candidate'] == 1])
else:
    print('Нет. В витрине дашборда типа карты нет.')
print('=== можно спутать ===')
if conf_cols:
    display(col_audit[col_audit['column_name'].astype(str).str.lower().isin(CONFUSABLE)])
else:
    print('—')
print('=== все колонки ===')
display(datamart_cols)


## 3) Зерно витрины и разбивка, которая на ней возможна

`final_df` = договор × месяц. Разбивка Visa/MC/МИР на этой таблице **невозможна**, если нет колонки типа карты.


In [ ]:
work = fd.copy()
month_col = next((c for c in ('report_month', 'snapshot_month_start') if c in work.columns), None)
if month_col:
    work['report_month'] = work[month_col].map(normalize_month)

key_cols = [c for c in ('inn', 'agr_id', 'contract_number') if c in work.columns]
print('source_kind=', source_kind, 'grain hint=', key_cols + (['report_month'] if month_col else []))
print('rows=', f'{len(work):,}')
if month_col:
    by_month = (
        work.groupby('report_month', dropna=False)
        .agg(
            rows=('report_month', 'size'),
            inns=('inn', 'nunique') if 'inn' in work.columns else ('report_month', 'size'),
            agrs=('agr_id', 'nunique') if 'agr_id' in work.columns else ('report_month', 'size'),
        )
        .reset_index()
    )
    print('=== договоры витрины по месяцам (периметр дашборда) ===')
    display(by_month)

# Если вдруг колонка типа карты уже есть — вот разбивка для дашборда.
metric_candidates = [
    c for c in (
        'trx_sum', 'trx_cnt', 'commission_from_ops', 'commission_total',
        'commission_monthly', 'chod', 'fin_result',
    ) if c in work.columns
]
breakdowns = {}
if card_cols:
    for col in card_cols:
        tmp = work.copy()
        tmp[col] = tmp[col].astype(str).str.strip()
        tmp['brand'] = tmp[col].map(classify_brand)
        agg = {'rows': (col, 'size')}
        for m in metric_candidates:
            tmp[m] = pd.to_numeric(tmp[m], errors='coerce')
            agg[m] = (m, 'sum')
        g = tmp.groupby(['brand', col], dropna=False).agg(**agg).reset_index()
        breakdowns[col] = g.sort_values('rows', ascending=False)
        print(f'=== разбивка витрины по {col} ===')
        display(breakdowns[col].head(40))
else:
    print('Разбивки по типу карты нет: колонки нет.')
    if 'business_cards' in work.columns:
        bc = pd.to_numeric(work['business_cards'], errors='coerce').fillna(0)
        print(
            'Флаг business_cards=1: '
            f'{int((bc == 1).sum()):,} строк '
            f'({100.0 * (bc == 1).mean():.1f}%). Это не Visa/MC/МИР.'
        )
    if 'mcc' in work.columns:
        print('Колонка mcc есть — MCC мерчанта, не тип карты. Для MCC уже отдельная таблица.')


## 4) Паттерн для дашборда: как уже сделали MCC

Тип карты — такое же trx-измерение, как MCC. На дашборде MCC живёт **не** в `final_df`, а в  
`sbx_da.tmp_shestopalov_acq_mcc_month` (`vd_acq_mcc_month.sql`).

Если этой таблицы нет — тип карты тем более нельзя взять из витрины договоров.


In [ ]:
mcc_cols = pd.DataFrame()
mcc_sample = pd.DataFrame()
mcc_exists = False
if drp_ok:
    schema, table = DRP_MCC_TABLE.split('.', 1)
    mcc_hit = fetch_drp(
        f'''
        SELECT table_schema, table_name
        FROM information_schema.tables
        WHERE lower(table_schema) = lower('{schema}')
          AND lower(table_name) = lower('{table}')
        ''',
        'mcc table exists',
        allow_fail=True,
    )
    mcc_exists = mcc_hit is not None and len(mcc_hit) > 0
    print('MCC month table exists:', mcc_exists, DRP_MCC_TABLE)
    if mcc_exists:
        mcc_cols = fetch_drp(
            f'''
            SELECT column_name, data_type
            FROM information_schema.columns
            WHERE lower(table_schema) = lower('{schema}')
              AND lower(table_name) = lower('{table}')
            ORDER BY ordinal_position
            ''',
            'mcc columns',
        )
        mcc_sample = fetch_drp(
            f'SELECT * FROM {DRP_MCC_TABLE} LIMIT 8',
            'mcc sample',
        )
        display(mcc_cols)
        display(mcc_sample)
        print('Для типов карт нужен такой же слой: report_month × card_type × trx_cnt/trx_sum/commission.')
else:
    print('SKIP проверка MCC-таблицы: нет DRP. Паттерн всё равно тот же, что vd_acq_mcc_month.sql')


## 5) Опционально: справочник FIID (маленький, без trx)

Отвечает только: «в озере бренды карт вообще есть?».  
Это не разбивка дашборда и не скан миллиарда транзакций.


In [ ]:
catalog = pd.DataFrame()
brand_catalog = pd.DataFrame()
if not RUN_FIID_CATALOG:
    print('SKIP FIID catalog')
else:
    if 'imp' in globals() and imp is not None:
        print('Reuse existing Impala')
    else:
        imp = connect(
            to='IMPALA',
            extra_options={'db': 'sandbox_ai'},
            driver_args={'tez.queue.name': 'ai'},
            kerberos={
                'keytab_path': '/home/jovyan/test_requests/tech.keytab',
                'use_credentials': True,
                'update_keytab': True,
            },
            user_params={'user_name': 'Shestopalov-VYur'},
        )
        imp._init_connection()
        print('Impala connected')

    catalog = fetch_imp(
        f'''
        SELECT
          cast(c_fiid as string) AS c_fiid,
          cast(c_fiid_grp as string) AS c_fiid_grp,
          cast(c_fiid_desc as string) AS c_fiid_desc
        FROM {FIID_TABLE}
        ''',
        'fiid catalog',
        allow_fail=True,
    )
    if catalog is None or catalog.empty:
        print('Справочник FIID не прочитался. На разбивку витрины это не влияет.')
        catalog = pd.DataFrame()
    else:
        catalog['brand'] = catalog['c_fiid_desc'].map(classify_brand)
        brand_catalog = (
            catalog.groupby('brand', dropna=False)
            .agg(fiid_cnt=('c_fiid', 'nunique'), desc_cnt=('c_fiid_desc', 'nunique'))
            .reset_index()
            .sort_values('fiid_cnt', ascending=False)
        )
        print('=== бренды в справочнике (не в витрине) ===')
        display(brand_catalog)
        examples = (
            catalog.groupby('brand', as_index=False)
            .agg(sample_desc=('c_fiid_desc', lambda s: ' | '.join(sorted(set(map(str, s)))[:5])))
        )
        display(examples)


## 6) VERDICT


In [ ]:
known = {'Visa', 'Mastercard', 'МИР', 'UnionPay', 'JCB', 'Amex'}
catalog_brands = set()
if len(brand_catalog):
    catalog_brands = set(brand_catalog.loc[brand_catalog['brand'].isin(known), 'brand'])

if card_cols:
    verdict = (
        f'на витрине уже есть колонка типа карты: {", ".join(card_cols)}. '
        'Разбивку можно вешать на дашборд с этой таблицы.'
    )
elif mcc_exists:
    verdict = (
        'на final_df типа карты нет (зерно договор×месяц). '
        'На дашборд — отдельная предрасчётная таблица по образцу tmp_shestopalov_acq_mcc_month, '
        'не живой скан scd1_trx.'
    )
else:
    verdict = (
        'на final_df типа карты нет. Живой scd1_trx на дашборд нельзя. '
        'Нужен слой report_month × card_type, как MCC.'
    )

print('=== VERDICT ===')
print(verdict)
print('source=', source_kind, 'card_cols=', card_cols or '—')
print('confusable=', conf_cols or '—')
print('mcc_month_table=', mcc_exists)
print('fiid catalog brands=', sorted(catalog_brands) or '—')
print()
print('Канон, когда будете собирать слой: fiids.c_fiid_desc по trx.c_fiid_iss (эмитент), не c_fiid_acq.')

out_xlsx = OUT_DIR / 'qc_card_types_datamart.xlsx'
with pd.ExcelWriter(out_xlsx, engine='openpyxl') as w:
    col_audit.to_excel(w, sheet_name='datamart_columns', index=False)
    if month_col:
        by_month.to_excel(w, sheet_name='datamart_by_month', index=False)
    for name, dfb in breakdowns.items():
        dfb.to_excel(w, sheet_name=f'by_{name}'[:31], index=False)
    if len(brand_catalog):
        brand_catalog.to_excel(w, sheet_name='fiid_catalog_brand', index=False)
    if len(mcc_cols):
        mcc_cols.to_excel(w, sheet_name='mcc_table_cols', index=False)
    pd.DataFrame([{
        'verdict': verdict,
        'source_kind': source_kind,
        'datamart': DRP_DATAMART,
        'rows': len(work),
        'card_cols': ', '.join(card_cols),
        'confusable': ', '.join(conf_cols),
        'mcc_exists': mcc_exists,
        'catalog_brands': ', '.join(sorted(catalog_brands)),
        'rev': NOTEBOOK_REV,
    }]).to_excel(w, sheet_name='verdict', index=False)
print('Saved', out_xlsx)
